# Field Scan : modèle maladies v3 (Kaggle)

Même recette que le modèle v2 retenu : ConvNeXt-Tiny pré-entraîné sur ImageNet, Pérou tiré 3 fois plus souvent, recompression JPEG, lissage des étiquettes 0,1, **moyenne de 3 modèles**, 4 vues, 6 époques (dernière époque gardée).

Ce qui change : la **Tanzanie 2025 passe en apprentissage** (11 766 photos de plus, autre téléphone, feuilles entières sur fond de champ). Il n'y a donc plus de validation : les seuils sont **repris de la v2**, réglés sur la Tanzanie 2025 quand elle servait de validation (alerte 0,30, nom 0,50, « je ne sais pas » sous 0,646). 900 photos de Tanzanie 2025 (300 par classe) servent seulement à suivre l'entraînement : elles sont apprises, leurs scores ne veulent rien dire.

| Rôle | Jeu |
| --- | --- |
| Apprentissage | Tanzanie 2023 + Pérou (saines et Sigatoka) + Tanzanie 2025 |
| Test | Bangladesh (Cordana et Pestalotiopsis jamais vues) |

Le test est lancé directement (`LANCER_TEST = True`) : ce modèle v3 est livré quel que soit son score au Bangladesh.

**Avant de lancer :** Settings → Accelerator → **GPU T4**, Internet **On**, le même jeu `feuilles-v2` que pour la v2 dans Input. **Ne rien modifier**, puis **Save Version → Save & Run All (Commit)**. Compter 2 à 3 h.


In [ ]:
# 1. GPU et emplacement des données
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path

trouves = sorted(Path('/kaggle/input').rglob('images.csv'))
assert trouves, "images.csv introuvable : ajouter le jeu 'feuilles' dans Input"
RACINE = trouves[0].parent
print('Données :', RACINE)
print(sorted(p.name for p in RACINE.iterdir()))

In [ ]:
# 2. Réglages (à changer d'un essai à l'autre)
import random, json, time
import numpy as np, pandas as pd, torch

RESEAU = 'convnext_tiny'            # réseau retenu en v2
POIDS_PEROU = 3                     # une photo du Pérou est tirée 3 fois plus souvent (le jeu est petit)
RECOMPRESSION_JPEG = True           # recompression JPEG au hasard (qualité 30 à 95), comme en v2
LISSAGE_ETIQUETTES = 0.1            # comme en v2
NB_MODELES = 3                      # moyenne de 3 modèles (graines différentes), comme en v2
NOM_ESSAI = (f'v3_{RESEAU}_perou{POIDS_PEROU}' + ('_jpeg' if RECOMPRESSION_JPEG else '')
             + (f'_lissage{LISSAGE_ETIQUETTES}' if LISSAGE_ETIQUETTES else '')
             + (f'_x{NB_MODELES}' if NB_MODELES > 1 else ''))   # nom du dossier de résultats
CLASSES = ['saine', 'black_sigatoka', 'fusarium']
EPOQUES = 6                         # le modèle gardé est celui de la dernière époque
# ConvNeXt-Tiny est 6 fois plus gros : lots plus petits (mémoire du GPU) et apprentissage plus doux
TAILLE_LOT = 32 if RESEAU == 'convnext_tiny' else 64
TAUX_APPRENTISSAGE = 1e-4 if RESEAU == 'convnext_tiny' else 3e-4
COUT_RATEE = 2                      # rater une malade compte 2 fois plus qu'une fausse alerte
OBJECTIF_NOM_JUSTE = 0.95           # étape 2 : quand la maladie est nommée, elle doit être juste 95 % du temps
ACCEPTATION_VALIDATION = 0.95       # seuil « inconnu » : garder 95 % des photos de validation
SIMULER_INCONNUES = False           # True : rechoisir la méthode « inconnu » avec des maladies retirées (2 petits entraînements)
METHODE_REJET_PAR_DEFAUT = 'confiance'   # méthode retenue par la simulation du 01/11 (la distance faisait moins bien que le hasard)
EPOQUES_SIMULATION = 3
LANCER_TEST = True                  # v3 : test direct, le modèle est livré quel que soit son score
# Seuils repris du modèle v2 (modeles/maladies/seuils.json), réglés sur la Tanzanie 2025 quand elle était la validation
SEUILS_V2 = {'seuil_alerte': 0.3, 'seuil_nom': 0.5, 'seuil_confiance': 0.645875871181488}
GRAINE = 0

random.seed(GRAINE); np.random.seed(GRAINE); torch.manual_seed(GRAINE)
appareil = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Calcul sur :', appareil, torch.cuda.get_device_name(0) if appareil == 'cuda' else '')
SORTIE = Path('/kaggle/working') / NOM_ESSAI
SORTIE.mkdir(parents=True, exist_ok=True)

In [ ]:
# 3. Découpage : par pays et par campagne, jamais au hasard
images = pd.read_csv(RACINE / 'images.csv')

def lire_liste(nom):
    # Entrées comparées sans extension : 'HEALTHY_1/Image_1094'
    return {l.strip().rsplit('.', 1)[0] for l in open(RACINE / nom, encoding='utf-8')
            if l.strip() and not l.startswith('#')}

def source_courte(chemin):
    # 'tanzanie_2023_dataverse/HEALTHY_1/HEALTHY_1/Image_1094.jpg' -> 'HEALTHY_1/Image_1094'
    return '/'.join(chemin.split('/')[-2:]).rsplit('.', 1)[0]

courte = images.source.map(source_courte)
tanz = images.jeu.str.startswith('tanzanie')
retire = (tanz & courte.isin(lire_liste('doublons_tanzanie.txt'))) | \
         ((images.jeu == 'bangladesh') & courte.isin(lire_liste('doublons_bangladesh.txt')))
print('Copies écartées :', images[retire].jeu.value_counts().to_dict())
images = images[~retire]

entrainement = images[images.jeu.isin(['tanzanie_2023', 'perou', 'tanzanie_2025'])]   # v3 : la Tanzanie 2025 passe en apprentissage
# Plus de validation : 300 photos de Tanzanie 2025 par classe servent seulement à suivre l'entraînement
# (elles sont apprises, leurs scores ne veulent rien dire)
validation = images[images.jeu == 'tanzanie_2025'].groupby('classe').sample(300, random_state=0)
test = images[images.jeu == 'bangladesh']

for nom, df in [('entraînement', entrainement), ('validation', validation), ('test', test)]:
    print(f'{nom:13s} {len(df):6d} images  ', df.groupby(['jeu', 'classe']).size().to_dict())
assert len(test) == 893, 'le test doit compter 893 images après retrait des doublons'
assert set(entrainement.classe) <= set(CLASSES) and set(validation.classe) <= set(CLASSES)

In [ ]:
# 4. Chargement des images ; augmentations fortes pour l'entraînement seulement
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T

NORMALISATION = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # celle d'ImageNet
import io

class JPEGAleatoire:
    # Réenregistre la photo en JPEG à une qualité tirée au hasard, une fois sur deux
    def __init__(self, p=0.5, qualites=(30, 95)):
        self.p, self.qualites = p, qualites
    def __call__(self, image):
        if random.random() >= self.p:
            return image
        tampon = io.BytesIO()
        image.save(tampon, format='JPEG', quality=random.randint(*self.qualites))
        return Image.open(tampon).convert('RGB')

aug_entrainement = T.Compose(([JPEGAleatoire()] if RECOMPRESSION_JPEG else []) + [
    T.RandomResizedCrop(224, scale=(0.4, 1.0), ratio=(0.8, 1.25)),   # zoom et cadrage variés
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(30),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.04),  # autre lumière, autre téléphone
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.3),        # photo un peu floue
    T.ToTensor(), NORMALISATION,
    T.RandomErasing(p=0.5, scale=(0.02, 0.15)),   # une zone masquée : le modèle ne peut pas compter sur un seul détail
])
aug_evaluation = T.Compose([T.ToTensor(), NORMALISATION])

class Feuilles(Dataset):
    def __init__(self, df, transformation, classes=CLASSES):
        self.chemins = list(df.image)
        self.etiquettes = [classes.index(c) if c in classes else -1 for c in df.classe]
        self.transformation = transformation
    def __len__(self):
        return len(self.chemins)
    def __getitem__(self, i):
        image = Image.open(RACINE / self.chemins[i]).convert('RGB')
        return self.transformation(image), self.etiquettes[i]

poids = np.where(entrainement.jeu == 'perou', POIDS_PEROU, 1.0)
tirage = WeightedRandomSampler(torch.tensor(poids, dtype=torch.double), num_samples=len(entrainement), replacement=True)
charge_entrainement = DataLoader(Feuilles(entrainement, aug_entrainement), TAILLE_LOT, sampler=tirage, num_workers=4)
charge_entrainement_sans_aug = DataLoader(Feuilles(entrainement, aug_evaluation), TAILLE_LOT, num_workers=4)
charge_validation = DataLoader(Feuilles(validation, aug_evaluation), TAILLE_LOT, num_workers=4)
charge_test = DataLoader(Feuilles(test, aug_evaluation), TAILLE_LOT, num_workers=4)
print(f"Part du Pérou dans un tour d'entraînement : {poids[entrainement.jeu.values == 'perou'].sum() / poids.sum():.0%}")

In [ ]:
# 5. Modèle pré-entraîné sur ImageNet, dernière couche remplacée
#    Toutes les photos restent en 224 × 224 px, quel que soit le réseau, pour comparer à égalité.
#    construire(n) renvoie le réseau, resume(x) (les nombres qui résument la photo, utilisés
#    pour la distance « inconnu ») et tete(f) (la dernière couche, qui donne un score par classe).
from torchvision import models

def construire(n_classes):
    if RESEAU == 'mobilenet_v3_large':
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        m.classifier[3] = torch.nn.Linear(m.classifier[3].in_features, n_classes)
        resume = lambda x: m.classifier[:3](torch.flatten(m.avgpool(m.features(x)), 1))
        tete = lambda f: m.classifier[3](f)
    elif RESEAU.startswith('efficientnet_b'):
        poids = {'efficientnet_b0': models.EfficientNet_B0_Weights.IMAGENET1K_V1,
                 'efficientnet_b1': models.EfficientNet_B1_Weights.IMAGENET1K_V1,
                 'efficientnet_b2': models.EfficientNet_B2_Weights.IMAGENET1K_V1}[RESEAU]
        m = getattr(models, RESEAU)(weights=poids)
        m.classifier[1] = torch.nn.Linear(m.classifier[1].in_features, n_classes)
        resume = lambda x: torch.flatten(m.avgpool(m.features(x)), 1)
        tete = lambda f: m.classifier(f)
    elif RESEAU == 'convnext_tiny':
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        m.classifier[2] = torch.nn.Linear(m.classifier[2].in_features, n_classes)
        resume = lambda x: m.classifier[:2](m.avgpool(m.features(x)))
        tete = lambda f: m.classifier[2](f)
    else:
        raise ValueError(RESEAU)
    return m.to(appareil), resume, tete

print(RESEAU, f': {sum(p.numel() for p in construire(len(CLASSES))[0].parameters()) / 1e6:.1f} millions de paramètres par modèle'
      f' ; {NB_MODELES} modèle(s) entraîné(s)')

In [ ]:
# 6. Entraînement ; on garde le modèle de la dernière époque
#    Le pas d'apprentissage descend jusqu'à presque zéro : le modèle final est le plus stable.
#    Choisir l'époque sur la validation donnait un modèle à moitié appris (le coût ne regarde que
#    saine / malade, pas le nom de la maladie) et dépendait du hasard d'une époque à l'autre.
#    Le coût et la F1 de chaque époque restent affichés, pour information.
from sklearn.metrics import f1_score

SEUILS = np.round(np.arange(0.05, 0.96, 0.05), 2)
VUES = [lambda x: x, lambda x: x.flip(3), lambda x: x.flip(2), lambda x: x.rot90(1, (2, 3))]

def predire(charge, quatre_vues, reseau):
    # Probabilités, vraies classes et résumés d'UN réseau. Avec quatre_vues : moyenne sur la photo, ses 2 miroirs et sa rotation.
    m, resume_, tete_ = reseau
    m.eval(); probas, vraies, resumes = [], [], []
    vues = VUES if quatre_vues else VUES[:1]
    with torch.no_grad():
        for x, y in charge:
            x = x.to(appareil); p, f = 0, 0
            for vue in vues:
                fv = resume_(vue(x)); p = p + torch.softmax(tete_(fv), 1); f = f + fv
            probas.append((p / len(vues)).cpu()); resumes.append((f / len(vues)).cpu()); vraies.append(y)
    return torch.cat(probas).numpy(), torch.cat(vraies).numpy(), torch.cat(resumes).numpy()

def predire_tous(charge, quatre_vues):
    # Moyenne des probabilités de tous les modèles gardés (un seul si NB_MODELES = 1).
    # Les résumés viennent du premier modèle (ils ne servent qu'à la distance, non utilisée avec la méthode « confiance »).
    resultats = [predire(charge, quatre_vues, r) for r in reseaux]
    return np.mean([p for p, _, _ in resultats], 0), resultats[0][1], resultats[0][2]

def entrainer(m, charge, epoques, apres_epoque):
    optimiseur = torch.optim.AdamW(m.parameters(), lr=TAUX_APPRENTISSAGE, weight_decay=1e-4)
    planning = torch.optim.lr_scheduler.CosineAnnealingLR(optimiseur, T_max=epoques)
    perte = torch.nn.CrossEntropyLoss(label_smoothing=LISSAGE_ETIQUETTES)
    for epoque in range(1, epoques + 1):
        m.train(); debut, total = time.time(), 0
        for x, y in charge:
            x, y = x.to(appareil), y.to(appareil)
            optimiseur.zero_grad(); l = perte(m(x), y); l.backward(); optimiseur.step()
            total += l.item() * len(y)
        planning.step()
        perte_moyenne = total / len(charge.dataset)
        assert np.isfinite(perte_moyenne), 'perte NaN : entraînement à relancer'
        apres_epoque(epoque, perte_moyenne, time.time() - debut)

def tableau_seuils(probas, vraies):
    # Pour chaque seuil d'alerte : malades ratées (dites saines), fausses alertes, coût
    malade, p_malade = vraies > 0, probas[:, 1:].sum(1)
    lignes = []
    for s in SEUILS:
        dit_malade = p_malade >= s
        ratees, fausses = int((~dit_malade & malade).sum()), int((dit_malade & ~malade).sum())
        lignes.append({'seuil': s, 'malades_ratees': ratees, 'fausses_alertes': fausses,
                       'rappel': round(dit_malade[malade].mean(), 3), 'cout': COUT_RATEE * ratees + fausses})
    return pd.DataFrame(lignes)

historique, reseaux = [], []
for k in range(NB_MODELES):
    graine = GRAINE + k
    random.seed(graine); np.random.seed(graine); torch.manual_seed(graine)
    reseau = construire(len(CLASSES))
    print(f'--- modèle {k + 1}/{NB_MODELES} (graine {graine})')

    def apres_epoque(epoque, perte_moyenne, duree):
        probas, vraies, _ = predire(charge_validation, False, reseau)   # une seule vue : plus rapide
        f1 = f1_score(vraies, probas.argmax(1), average='macro')
        cout = int(tableau_seuils(probas, vraies).cout.min())
        historique.append({'modele': k + 1, 'epoque': epoque, 'perte': perte_moyenne, 'f1_validation': f1, 'cout_validation': cout})
        print(f'époque {epoque}  perte {perte_moyenne:.3f}  F1 3 classes {f1:.3f}  coût {cout}  ({duree:.0f} s)')

    entrainer(reseau[0], charge_entrainement, EPOQUES, apres_epoque)
    torch.save(reseau[0].state_dict(), SORTIE / f'maladies_v3_{RESEAU}_{k + 1}.pt')
    reseaux.append(reseau)
modele, resume, tete = reseaux[0]   # le premier modèle sert aux cartes d'attention
print(f'Modèle(s) gardé(s) : époque {EPOQUES} (la dernière)')

In [ ]:
# 7. v3 : rien n'est choisi ici. Mêmes réglages que la v2 (moyenne des 3 modèles, 4 vues) et seuils repris de la v2.
#    Les photos de contrôle sont apprises : leurs scores servent seulement à vérifier que rien n'a cassé.
QUATRE_VUES = True
cout_un_modele = cout_moyenne = None
probas_val, vraies_val, resumes_val = predire_tous(charge_validation, True)
tableau_val = tableau_seuils(probas_val, vraies_val)
tableau_1 = tableau_4 = tableau_val
malade_val = vraies_val > 0
p_malade_val = probas_val[:, 1:].sum(1)
SEUIL_ALERTE = SEUILS_V2['seuil_alerte']
print(f"Modèles moyennés : {len(reseaux)} ; 4 vues ; seuil d'alerte repris de la v2 : {SEUIL_ALERTE}")


In [ ]:
# 8. Étape 2 : seuil pour nommer la maladie, repris de la v2
SEUIL_NOM = SEUILS_V2['seuil_nom']
print(f'Seuil pour nommer la maladie (v2) : {SEUIL_NOM}')


In [ ]:
# 9. Inconnues simulées : quelle méthode repère le mieux une maladie jamais vue ?
#    On entraîne un modèle sans l'une des deux maladies, puis on la lui montre comme une « inconnue ».
#    Pour chaque méthode (confiance, distance, les deux), le seuil garde ACCEPTATION_VALIDATION des photos
#    connues de la validation ; on mesure la part de la maladie retirée qui est rejetée.
#    Aucune photo du Bangladesh n'est utilisée. Une seule vue et EPOQUES_SIMULATION époques : plus rapide.
from sklearn.covariance import LedoitWolf
from sklearn.metrics import roc_auc_score

def statistiques_distance(resumes, etiquettes, n_classes):
    centres = np.stack([resumes[etiquettes == k].mean(0) for k in range(n_classes)])
    precision = LedoitWolf().fit(resumes - centres[etiquettes]).precision_.astype(np.float32)
    return centres, precision

def distance_a(resumes, centres, precision):
    # distance de Mahalanobis au centre de la classe la plus proche
    return np.min([np.einsum('ij,jk,ik->i', resumes - c, precision, resumes - c) for c in centres], 0)

def regler_rejet(conf, dist, methode):
    # renvoie (seuil de confiance, seuil de distance) qui gardent ACCEPTATION_VALIDATION des photos
    garde = ACCEPTATION_VALIDATION
    if methode == 'confiance':
        return float(np.quantile(conf, 1 - garde)), np.inf
    if methode == 'distance':
        return -np.inf, float(np.quantile(dist, garde))
    bas, haut = 0.0, 1 - garde        # les deux : même part q pour chaque mesure
    for _ in range(40):
        q = (bas + haut) / 2
        rejet = (conf < np.quantile(conf, q)) | (dist > np.quantile(dist, 1 - q))
        bas, haut = (q, haut) if rejet.mean() < 1 - garde else (bas, q)
    return float(np.quantile(conf, q)), float(np.quantile(dist, 1 - q))

def rejete(conf, dist, seuils):
    return (conf < seuils[0]) | (dist > seuils[1])

METHODES = ['confiance', 'distance', 'les_deux']
simulation = None
if SIMULER_INCONNUES:
    lignes = []
    for retiree in ['black_sigatoka', 'fusarium']:
        gardees = [c for c in CLASSES if c != retiree]
        ent = entrainement[entrainement.classe != retiree]
        p = np.where(ent.jeu == 'perou', POIDS_PEROU, 1.0)
        tir = WeightedRandomSampler(torch.tensor(p, dtype=torch.double), num_samples=len(ent), replacement=True)
        reseau_sim = construire(len(gardees))
        print(f'--- sans {retiree} : entraînement sur {gardees}')
        entrainer(reseau_sim[0], DataLoader(Feuilles(ent, aug_entrainement, gardees), TAILLE_LOT, sampler=tir, num_workers=4),
                  EPOQUES_SIMULATION, lambda e, l, d: print(f'  époque {e}  perte {l:.3f}  ({d:.0f} s)'))
        _, et_ent, res_ent = predire(DataLoader(Feuilles(ent, aug_evaluation, gardees), TAILLE_LOT, num_workers=4),
                                     False, reseau_sim)
        centres_s, precision_s = statistiques_distance(res_ent, et_ent, len(gardees))
        pr, _, res = predire(charge_validation, False, reseau_sim)
        conf, dist = pr.max(1), distance_a(res, centres_s, precision_s)
        inconnue = (validation.classe == retiree).values
        print(f'  AUROC confiance {roc_auc_score(inconnue, -conf):.3f}   AUROC distance {roc_auc_score(inconnue, dist):.3f}'
              '   (1 = séparation parfaite, 0,5 = hasard)')
        for methode in METHODES:
            r = rejete(conf, dist, regler_rejet(conf[~inconnue], dist[~inconnue], methode))
            lignes.append({'maladie_retiree': retiree, 'methode': methode,
                           'inconnue_rejetee': round(r[inconnue].mean(), 3),
                           'connues_rejetees': round(r[~inconnue].mean(), 3)})
        del reseau_sim; torch.cuda.empty_cache()
    simulation = pd.DataFrame(lignes)
    print(simulation.to_string(index=False))
    METHODE_REJET = simulation.groupby('methode').inconnue_rejetee.mean().idxmax()
else:
    METHODE_REJET = METHODE_REJET_PAR_DEFAUT
print('\nMéthode retenue pour « inconnu » :', METHODE_REJET)

In [ ]:
# 10. « Inconnu » : confiance seule (méthode de la v2), seuil repris de la v2
centres = precision_cov = None

def distance(resumes):
    return np.zeros(len(resumes))

SEUIL_CONFIANCE, SEUIL_DISTANCE = SEUILS_V2['seuil_confiance'], np.inf
print(f'Méthode : confiance ; seuil repris de la v2 : {SEUIL_CONFIANCE:.3f}')


In [ ]:
# 11. Décision complète et bilan de validation
#     Réponses possibles : saine, black_sigatoka, fusarium, malade_non_identifiee, inconnu
def decider(probas, resumes):
    p_malade = probas[:, 1:].sum(1)
    part = probas[:, 1:].max(1) / np.maximum(p_malade, 1e-9)
    nom = np.array(CLASSES)[1 + probas[:, 1:].argmax(1)]
    reponse = np.where(p_malade < SEUIL_ALERTE, 'saine',
                       np.where(part >= SEUIL_NOM, nom, 'malade_non_identifiee')).astype(object)
    reponse[rejete(probas.max(1), distance(resumes), (SEUIL_CONFIANCE, SEUIL_DISTANCE))] = 'inconnu'
    return reponse

ORDRE = ['saine', 'black_sigatoka', 'fusarium', 'malade_non_identifiee', 'inconnu']
MALADE = ['black_sigatoka', 'fusarium', 'malade_non_identifiee']
reponse_val = decider(probas_val, resumes_val)
reel_val = np.array(CLASSES)[vraies_val]

print('Validation (lignes : classe réelle, colonnes : réponse)')
tab = pd.crosstab(pd.Series(reel_val, name='réel'), pd.Series(reponse_val, name='réponse'))
print(tab.reindex(columns=[c for c in ORDRE if c in tab.columns]))

saine_v = reel_val == 'saine'
bilan_val = {
    'malades_dites_saines': int(((reponse_val == 'saine') & ~saine_v).sum()),
    'malades_signalees': float(np.isin(reponse_val[~saine_v], MALADE).mean()),
    'malades_inconnu': float((reponse_val[~saine_v] == 'inconnu').mean()),
    'fausses_alertes_saines': float(np.isin(reponse_val[saine_v], MALADE).mean()),
    'saines_inconnu': float((reponse_val[saine_v] == 'inconnu').mean()),
    'nom_juste_quand_nomme': float(np.mean([r == v for r, v in zip(reponse_val, reel_val)
                                            if r in ('black_sigatoka', 'fusarium')])),
    'part_non_identifiee': float((reponse_val == 'malade_non_identifiee').mean()),
}
print(json.dumps(bilan_val, indent=2))

In [ ]:
# 12. Cartes d'attention (Grad-CAM) : où le modèle regarde-t-il pour décider ?
#     Rouge = zones qui ont le plus pesé dans la réponse. 4 photos de validation par classe, tirées au hasard,
#     plus 4 Fusarium prises pour de la Black Sigatoka. Enregistré dans cartes_attention.png.
#     Avec plusieurs modèles, les cartes sont celles du premier.
import matplotlib.pyplot as plt

memo = {}
def garder(module, entree, sortie):
    memo['activations'] = sortie
    sortie.register_hook(lambda g: memo.__setitem__('gradients', g))
crochet = modele.features[-1].register_forward_hook(garder)

def carte_attention(x):
    modele.eval(); modele.zero_grad()
    with torch.enable_grad():
        score = modele(x[None].to(appareil))
        k = int(score.argmax(1))
        score[0, k].backward()
    poids = memo['gradients'].mean((2, 3), keepdim=True)
    c = torch.relu((poids * memo['activations']).sum(1, keepdim=True))
    c = torch.nn.functional.interpolate(c, size=x.shape[1:], mode='bilinear')[0, 0]
    return (c / (c.max() + 1e-9)).detach().cpu().numpy(), CLASSES[k]

hasard = np.random.default_rng(GRAINE)
groupes = [(c, np.flatnonzero(reel_val == c)) for c in CLASSES]
groupes.append(('fusarium dit black_sigatoka', np.flatnonzero((reel_val == 'fusarium') & (reponse_val == 'black_sigatoka'))))
jeu_val = Feuilles(validation, aug_evaluation)
moyenne, ecart = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(len(groupes), 4, figsize=(12, 3.3 * len(groupes)))
for ligne, (titre, indices) in enumerate(groupes):
    choisis = hasard.choice(indices, size=min(4, len(indices)), replace=False) if len(indices) else []
    for col in range(4):
        ax = axes[ligne, col]; ax.axis('off')
        if col >= len(choisis):
            continue
        x, _ = jeu_val[int(choisis[col])]
        c, rep = carte_attention(x)
        ax.imshow(np.clip(x.permute(1, 2, 0).numpy() * ecart + moyenne, 0, 1))
        ax.imshow(c, cmap='jet', alpha=0.4)
        ax.set_title(f'réel : {titre}\nmodèle : {rep}', fontsize=8)
crochet.remove()
plt.tight_layout(); plt.savefig(SORTIE / 'cartes_attention.png', dpi=110); plt.show()

In [ ]:
# 13. Résumé de l'essai, pour comparer les réseaux entre eux
#     Sur toute la validation, avec la décision complète ; puis sur les archives HEALTHY-1, BLACK SIGATOKA-1
#     et FUSARIUM WILT-1 avec la règle simple (malade si p_malade >= 0,5, sans « je ne sais pas »).
dossier = validation.source.str.split('/').str[1].values
sous = np.isin(dossier, ['HEALTHY-1', 'BLACK SIGATOKA-1', 'FUSARIUM WILT-1'])
regle_05 = np.array(CLASSES)[np.where(p_malade_val >= 0.5, 1 + probas_val[:, 1:].argmax(1), 0)]
fus = reel_val == 'fusarium'
resume_essai = {
    'reseau': RESEAU, 'poids_perou': POIDS_PEROU,
    'recompression_jpeg': RECOMPRESSION_JPEG, 'lissage_etiquettes': LISSAGE_ETIQUETTES,
    'millions_parametres': round(sum(p.numel() for p in modele.parameters()) / 1e6, 1),
    'nb_modeles_moyennes': len(reseaux), 'cout_1er_modele_seul': cout_un_modele, 'cout_moyenne': cout_moyenne,
    'cout_validation': int(tableau_val.cout.min()), 'seuil_alerte': SEUIL_ALERTE,
    'malades_dites_saines': bilan_val['malades_dites_saines'],
    'fusarium_dites_saines': int((fus & (reponse_val == 'saine')).sum()),
    'fausses_alertes_saines': round(bilan_val['fausses_alertes_saines'], 3),
    'nom_juste_quand_nomme': round(bilan_val['nom_juste_quand_nomme'], 3),
    'archives_1_regle_05_bien_classees': {c: int(((reel_val == c) & (regle_05 == c) & sous).sum()) for c in CLASSES},
    'archives_1_regle_05_fusarium_dites_saines': int((fus & (regle_05 == 'saine') & sous).sum()),
}
print(json.dumps(resume_essai, indent=2, ensure_ascii=False))

In [ ]:
# 13 bis. Pays jamais vu : si le modèle a été entraîné SANS le Pérou (POIDS_PEROU = 0), on le fait répondre
#         sur les saines et les Sigatoka du Pérou, avec les seuils réglés sur la Tanzanie. Rien n'est réglé ici.
evaluation_perou = None
if POIDS_PEROU == 0:
    perou = images[images.jeu == 'perou']
    probas_p, vraies_p, resumes_p = predire_tous(DataLoader(Feuilles(perou, aug_evaluation), TAILLE_LOT, num_workers=4), QUATRE_VUES)
    rep_p, reel_p = decider(probas_p, resumes_p), np.array(CLASSES)[vraies_p]
    saine_p, sig_p = reel_p == 'saine', reel_p == 'black_sigatoka'
    regle_05_p = probas_p[:, 1:].sum(1) >= 0.5
    evaluation_perou = {
        'saines_bien_reconnues': round(float((rep_p[saine_p] == 'saine').mean()), 3),
        'fausses_alertes_saines': round(float(np.isin(rep_p[saine_p], MALADE).mean()), 3),
        'saines_inconnu': round(float((rep_p[saine_p] == 'inconnu').mean()), 3),
        'sigatoka_signalee_malade': round(float(np.isin(rep_p[sig_p], MALADE).mean()), 3),
        'sigatoka_bien_nommee': round(float((rep_p[sig_p] == 'black_sigatoka').mean()), 3),
        'sigatoka_dite_saine': round(float((rep_p[sig_p] == 'saine').mean()), 3),
        'sigatoka_inconnu': round(float((rep_p[sig_p] == 'inconnu').mean()), 3),
        'regle_05_fausses_alertes_saines': round(float(regle_05_p[saine_p].mean()), 3),
        'regle_05_sigatoka_dite_saine': round(float((~regle_05_p[sig_p]).mean()), 3),
    }
    print(f'Pérou, pays jamais vu par ce modèle ({saine_p.sum()} saines, {sig_p.sum()} Sigatoka) :')
    print(json.dumps(evaluation_perou, indent=2))
else:
    print('Modèle entraîné avec le Pérou : pas d\'évaluation sur le Pérou.')

In [ ]:
# 14. Sauvegarde de l'essai (poids, seuils, statistiques pour le rejet, résultats de validation)
if centres is not None:
    np.savez(SORTIE / 'rejet_inconnu.npz', centres=centres, precision_cov=precision_cov)
reglages = {
    'nom_essai': NOM_ESSAI, 'reseau': RESEAU, 'classes': CLASSES, 'epoques': EPOQUES,
    'taux_apprentissage': TAUX_APPRENTISSAGE, 'poids_perou': POIDS_PEROU, 'cout_ratee': COUT_RATEE,
    'recompression_jpeg': RECOMPRESSION_JPEG, 'lissage_etiquettes': LISSAGE_ETIQUETTES,
    'historique': historique,
    'cout_1_vue': int(tableau_1.cout.min()), 'cout_4_vues': int(tableau_4.cout.min()), 'quatre_vues': QUATRE_VUES,
    'tableau_seuils': tableau_val.to_dict('records'),
    'seuil_alerte': SEUIL_ALERTE, 'seuil_nom': SEUIL_NOM,
    'simulation_inconnues': simulation.to_dict('records') if simulation is not None else None,
    'methode_rejet': METHODE_REJET, 'seuil_confiance': SEUIL_CONFIANCE, 'seuil_distance': SEUIL_DISTANCE,
    'validation': bilan_val, 'resume': resume_essai, 'evaluation_perou': evaluation_perou,
}
json.dump(reglages, open(SORTIE / 'resultats.json', 'w'), indent=2, ensure_ascii=False, default=float)
print('Écrit dans', SORTIE, ':', sorted(p.name for p in SORTIE.iterdir()))

## Test final sur le Bangladesh

Lancé directement dans ce notebook (`LANCER_TEST = True`) : le modèle v3 est livré quel que soit son score, rien n'est réglé d'après le Bangladesh.

In [ ]:
# 15. Test sur le Bangladesh : une seule fois, après tous les réglages
#     Sans LANCER_TEST, la cellule ne fait rien (et la version Kaggle se termine sans erreur, sorties gardées).
if not LANCER_TEST:
    print('Test non lancé : mettre LANCER_TEST = True seulement pour le test final')
else:

    probas_test, _, resumes_test = predire_tous(charge_test, QUATRE_VUES)
    t = test.assign(reponse=decider(probas_test, resumes_test), confiance=probas_test.max(1),
                    distance=distance(resumes_test), p_malade=probas_test[:, 1:].sum(1))

    # La sigatoka du Bangladesh compte comme Black Sigatoka (forme non précisée par le jeu)
    print('Test (lignes : classe réelle, colonnes : réponse)')
    tab = pd.crosstab(t.classe, t.reponse)
    print(tab.reindex(columns=[c for c in ORDRE if c in tab.columns]))

    sig, saines = t[t.classe == 'sigatoka'], t[t.classe == 'saine']
    inconnues = t[t.classe.isin(['cordana', 'pestalotiopsis'])]
    connues = t[t.classe.isin(['saine', 'sigatoka'])]
    est_inc = t.classe.isin(['cordana', 'pestalotiopsis'])
    resultats = {
        'sigatoka_nommee_black_sigatoka': float((sig.reponse == 'black_sigatoka').mean()),
        'sigatoka_signalee_malade': float(sig.reponse.isin(MALADE).mean()),
        'sigatoka_dite_saine': float((sig.reponse == 'saine').mean()),
        'sigatoka_inconnu': float((sig.reponse == 'inconnu').mean()),
        'saines_bien_reconnues': float((saines.reponse == 'saine').mean()),
        'fausses_alertes_sur_saines': float(saines.reponse.isin(MALADE).mean()),
        'inconnues_dites_inconnu': float((inconnues.reponse == 'inconnu').mean()),
        'inconnues_signalees_malade': float(inconnues.reponse.isin(MALADE).mean()),
        'inconnues_dites_saines': float((inconnues.reponse == 'saine').mean()),
        'connues_dites_inconnu': float((connues.reponse == 'inconnu').mean()),
        # Capacité à séparer connu / inconnu, sans seuil (1 = parfait, 0,5 = hasard)
        'auroc_inconnu_confiance': float(roc_auc_score(est_inc, -t.confiance)),
        'auroc_inconnu_distance': float(roc_auc_score(est_inc, t.distance)),
    }
    for k, v in resultats.items():
        print(f'{k:32s} {v:.3f}')

    reglages['test_bangladesh'] = resultats
    json.dump(reglages, open(SORTIE / 'resultats.json', 'w'), indent=2, ensure_ascii=False, default=float)
    t.to_csv(SORTIE / 'reponses_test.csv', index=False)
    print('Écrit dans', SORTIE)